In [ ]:
"""
================================================================================
TESIS MAGÍSTER EN ECONOMÍA UC
Contagio predictivo sectorial y NBFIs en Chile
================================================================================
Script: consolidado_data.py
Autor:  Carlos González
Fecha:  Abril 2026
Ultima actualizacion: Mayo 2026

Panel de datos consolidado - todas las bases del proyecto necesarias a priori.

ORDEN DE EJECUCIÓN:
    Paso 1  → Balances por sector (panel_balances.csv)
    Paso 2  → Operaciones por sector (panel_operaciones.csv)
    Paso 3  → Matriz who-to-whom completa (panel_matriz_completa.csv)
              Saldos (panel_matriz_saldos.csv)
              Flujos (panel_matriz_operaciones.csv)

    Paso 4  → Red financiera. Similar a matriz who-to-whom solamente que 
            excluye instrumento Dinero legal y redondea todo lo menor a 0 con 
            (lower=0) debido a posibles errores de redondeo BCCh (panel_red_saldos.csv)
            Esta base será clave para calcular todos los indicadores.

    Paso 5  → Base HHI fuentes sector real. HHI es el primer índice del trabajo de data
            Caracteriza la concentración en tenencias del sector j en el financiamiento total
            recibido/emitido por el sector i. En esaca 0 - 10.000: convención regulatoria
            bajo 1,500 es poco concentrado, 1,500–2,500 es moderado, sobre 2,500 es altamente concentrado.
             (panel_hhi_fuentes.csv) -> (^2) Eso hace que el índice sea mucho más sensible a la concentración
              que simplemente sumar las participaciones.  El HHI mide concentración de fuentes de financiamiento,
               es decir pasivos del sector real, no sus activos
    Paso 6  → HHI serie temporal (resumen_hhi_por_periodo.csv)

    Paso 7  → IS Upper-Worms adaptado + exposición bilateral NBFI→Bancos
              (panel_is_upper_worms.csv)
              IS_pasivo:             fracción del activo SF expuesta ante default de i como deudor
              IS_activo:             fracción del activo SF que depende de i como fuente
              exp_nbfi_en_bancos:    activos NBFI_i invertidos en instrumentos bancarios (MMMCL)
              ratio_exp_nbfi_bancos: exp_nbfi_en_bancos / activo_proxy_i → NaN para no-NBFI

    Paso 8  → Índice de dependencia de fondeo bilateral D^{fondeo}_{j<-i,t}
              (panel_dependencia_fondeo.csv)
              D_fondeo: cuota del fondeo del deudor j que proviene del acreedor i
              Sumas por (período, deudor) iguales a 1 por construcción.
              Conexión con HHI: HHI_{j,t} = sum_i (D_fondeo_{j<-i,t})^2 * 10000

    Paso 9  → Notional stocks à la Holm-Hadulla et al. (2023), deflactados por
              UF (estándar Papers BCCh).literatura DV también deflacta usualmente por pib
              a la Antonakakis et al. (2015), e input para VAR Diebold-Yilmaz.
              (panel_notional_stocks.csv, panel_notional_stocks_agregado.csv,
               panel_var_dy.csv)

              Construcción en dos arreglos complementarios:
                (a) Holm-Hadulla: usa operaciones (no saldos) acumuladas desde un
                    saldo base → elimina efecto precio (revalorización).
                (b) Antonakakis:  deflacta por UF cada flujo en su período y el
                    saldo base por UF inicial → elimina efecto inflación.

              Variables construidas (todas trazadas en el output):
                A_notional_nominal: nominal (efecto precio eliminado, inflación
                                    presente). Análogo a Holm-Hadulla puro.
                A_notional_real:    real (efecto precio e inflación eliminados).
                                    Es la variable usada en el VAR D-Y.
                d_log_A_real:       Δ log A_real, serie del VAR (incluye S121 después se ve en código DY si se incluye o no).

              Si el archivo de UF no está disponible, solo se construye la
              versión nominal y se emite un warning explícito.

FUENTE ÚNICA:
    Todos los indicadores de la Etapa 2 se construyen desde
    panel_matriz_saldos.csv (producido en el Paso 3). No hay
    pipelines paralelos ni re-lecturas del Excel original.
 
NOMBRES CANÓNICOS DE INSTRUMENTOS (usados en todo el pipeline):
    Cuotas FMM | Cuotas FNM | Depositos | Prestamos | Titulos deuda
    Dinero legal (excluido de la red pero presente en la base completa)
 
UNIDADES: miles de millones de pesos corrientes.
PERÍODO:  2003T1–2025T3 (91 trimestres).
================================================================================
"""
 
import pandas as pd
import numpy as np
from pathlib import Path
 
# ============================================================
# RUTAS — AJUSTA INPUT_DIR Y OUTPUT_DIR
# ============================================================
BASE_DIR  = Path("../..") # path base

INPUT_DIR  = BASE_DIR / "1_datos/0_data_original/1_CNSI"
OUTPUT_DIR = BASE_DIR / "1_datos/1_clean_data"
 
FILE_BALANCES    = INPUT_DIR / "Base_Balances_CNSI.xlsx"
FILE_OPERACIONES = INPUT_DIR / "Base_Operaciones_CNSI.xlsx"
FILE_MATRIZ      = INPUT_DIR / "Matriz_Exposiciones_CNSI.xlsx"

# Serie de UF al cierre de cada trimestre (input externo, opcional).
# Formato esperado: archivo xlsx descargado de la Base de Datos Estadísticos
# del Banco Central de Chile, sección "Indicadores de reajustabilidad,
# fin de mes" (UF, IVP, UTM), filtrado a frecuencia trimestral (marzo, junio,
# septiembre, diciembre = cierre de T1, T2, T3, T4).
# La hoja de datos se llama "Cuadro" con headers en fila 2 y datos desde fila 3.
# Si el archivo no existe o tiene otro formato, el Paso 9 se ejecuta solo
# en versión nominal con warning explícito.
FILE_UF          = INPUT_DIR / "UF_IVP_UTM_F.xlsx"
 
# ============================================================
# CONSTANTES
# ============================================================
 
PERIODO_INICIO = "2003T1"
PERIODO_FIN    = "2025T3"
 
SECTOR_MAP = {
    "BC":                      ("S121",      "Banco Central"),
    "Bancos":                  ("S122",      "Bancos y cooperativas"),
    "FMM":                     ("S123",      "Fondos mercado monetario"),
    "FNM":                     ("S124",      "Fondos mercado no monetario"),
    "Otros":                   ("S125_S126", "Otros intermediarios y auxiliares"),
    "Seguros":                 ("S128",      "Compañías de seguros"),
    "FP":                      ("S129",      "Fondos de pensiones"),
    "Gobierno_General":        ("S13",       "Gobierno general"),
    "Empresas_No_Financieras": ("S11",       "Empresas no financieras"),
    "Hogares":                 ("S14_S15",   "Hogares e IPSFLSH"),
    "Resto_Mundo":             ("S2",        "Resto del mundo"),
}
 
SECTOR_ORDER = ["S121","S122","S123","S124","S125_S126","S128","S129","S13","S11","S14_S15","S2"]
 
SF          = ["S121","S122","S123","S124","S125_S126","S128","S129"]
NBFI        = ["S123","S124","S125_S126","S128","S129"]
BANCARIO    = ["S122"]
SECTOR_REAL = ["S11","S14_S15"]
SECTOR_INTERES = ["S121","S122","S123","S124","S125_S126","S128","S129","S11","S14_S15","S2"]

# Instrumentos en la red (excluye Dinero legal)
INSTR_RED = ["Prestamos","Titulos deuda","Depositos","Cuotas FMM","Cuotas FNM"]
 
# Instrumentos para HHI fuentes sector real (crédito directo)
INSTR_HHI = ["Prestamos","Titulos deuda"]
 
NOMBRE_MAP = {
    "S121":"Banco Central","S122":"Bancos y cooperativas",
    "S123":"FMM","S124":"FNM","S125_S126":"Otros intermediarios",
    "S128":"Compañías de seguros","S129":"Fondos de pensiones",
    "S13":"Gobierno general","S11":"Empresas no financieras",
    "S14_S15":"Hogares","S2":"Resto del mundo",
}
 
CLASIF_MAP = {
    "S122":"Bancario","S123":"NBFI","S124":"NBFI",
    "S125_S126":"NBFI","S128":"NBFI","S129":"NBFI",
    "S121":"Banco Central","S13":"Gobierno",
    "S2":"Resto del mundo","S11":"Empresas NF","S14_S15":"Hogares",
}
 
# ============================================================
# MAPAS DE VARIABLES — BALANCES Y OPERACIONES
# ============================================================
 
ACTIVOS_MAP = {
    "1.2.1. Títulos de deuda a corto plazo activos": "act_titulos_cp",
    "1.2.2. Títulos de deuda a largo plazo activos": "act_titulos_lp",
    "1.2. Títulos activos":                          "act_titulos_total",
    "1.4.1. Préstamos a corto plazo activos":        "act_prestamos_cp",
    "1.4.2. Préstamos a largo plazo activos":        "act_prestamos_lp",
    "1.4. Préstamos activos":                        "act_prestamos_total",
    "1.5.2. Cuotas emitidas por fondos del mercado monetario activos":    "act_cuotas_fmm",
    "1.5.3. Cuotas emitidas por fondos del mercado no monetario activos": "act_cuotas_fnm",
    # Hogares: cuotas en 1.3.x
    "1.3.2. Cuotas emitidas por fondos del mercado monetario activos":    "act_cuotas_fmm",
    "1.3.3. Cuotas emitidas por fondos del mercado no monetario activos": "act_cuotas_fnm",
    "1.1.1. Efectivo y depósitos vista activos": "act_depositos_vista",
    "1.1.2. Otros depósitos activos":            "act_depositos_otros",
    "1. I. Activos Financieros":                 "act_total",
}
 
PASIVOS_MAP = {
    "3.2.1. Títulos de deuda a corto plazo pasivos": "pas_titulos_cp",
    "3.2.2. Títulos de deuda a largo plazo pasivos": "pas_titulos_lp",
    "3.2. Títulos de deuda pasivos":                 "pas_titulos_total",
    "3.3.1. Préstamos a corto plazo pasivos":        "pas_prestamos_cp",
    "3.3.2. Préstamos a largo plazo pasivos":        "pas_prestamos_lp",
    "3.3. Préstamos pasivos":                        "pas_prestamos_total",
    "3.1.1. Efectivo y depósitos vista pasivos":     "pas_depositos_vista",
    "3.1.2. Otros depósitos pasivos":                "pas_depositos_otros",
    "3.1. Efectivo y depósitos pasivos":             "pas_depositos_total",
    "3. III. Pasivos":                               "pas_total",
    "2. II. Activos Financieros netos (I-III)":      "activos_netos",
}
 
PASIVOS_MAP_HOGARES = {
    "3.1. Préstamos pasivos":                    "pas_prestamos_total",
    "3.1.1. Préstamos a corto plazo pasivos":    "pas_prestamos_cp",
    "3.1.2. Préstamos a largo plazo pasivos":    "pas_prestamos_lp",
    "3. III. Pasivos":                           "pas_total",
    "2. II. Activos Financieros Netos (I-III)":  "activos_netos",
}
 
PASIVOS_MAP_ENF = {
    "3.1. Títulos pasivos":                      "pas_titulos_total",
    "3.1.1. Títulos a corto plazo pasivos":      "pas_titulos_cp",
    "3.1.2. Títulos a largo plazo pasivos":      "pas_titulos_lp",
    "3.2. Préstamos pasivos":                    "pas_prestamos_total",
    "3.2.1. Préstamos a corto plazo pasivos":    "pas_prestamos_cp",
    "3.2.2. Préstamos a largo plazo pasivos":    "pas_prestamos_lp",
    "3. III. Pasivos":                           "pas_total",
    "2. II. Activos Financieros Netos (I-III)":  "activos_netos",
}
 
def _make_op_map(pmap):
    """Convierte mapa de balances a mapa de operaciones."""
    m = {
        k.replace("3. III. Pasivos", "3. III. Pasivos netos contraídos")
         .replace("2. II. Activos Financieros netos (I-III)",
                  "2. II. Capacidad (+) / Necesidad (-) de financiamiento (I-III)")
         .replace("2. II. Activos Financieros Netos (I-III)",
                  "2. II. Capacidad (+) / Necesidad (-) de financiamiento (I-III)"): v
        for k, v in pmap.items()
    }
    m["2. II. Capacidad (+) / Necesidad (-) de financiamiento (I-III)"] = "cap_nec_financiamiento"
    return m
 
OP_ACTIVOS_MAP     = dict(ACTIVOS_MAP)
OP_PASIVOS_MAP     = _make_op_map(PASIVOS_MAP)
OP_PASIVOS_HOG     = _make_op_map(PASIVOS_MAP_HOGARES)
OP_PASIVOS_ENF     = _make_op_map(PASIVOS_MAP_ENF)
 
# ============================================================
# FUNCIONES — PASOS 1 Y 2
# ============================================================
 
def _parse_periodo(v):
    if pd.isna(v): return None
    if isinstance(v, str):
        try: v = pd.to_datetime(v)
        except: return None
    return f"{v.year}T{(v.month-1)//3+1}"
 
def _get_q(v):
    if pd.isna(v): return None
    if isinstance(v, str): v = pd.to_datetime(v)
    return (v.month-1)//3+1
 
def _read_sector_sheet(filepath, sheet_name, amap, pmap):
    df_raw = pd.read_excel(filepath, sheet_name=sheet_name, header=None)
    # Normalizar espacios en headers (Empresas NF tiene doble espacio en préstamos CP)
    headers = [" ".join(str(h).split()) if pd.notna(h) else h for h in df_raw.iloc[2].tolist()]
    data = df_raw.iloc[3:].copy()
    data.columns = headers
    data = data.dropna(subset=[headers[0]])
    data = data.rename(columns={headers[0]: "fecha_raw"})
    data["periodo"]   = data["fecha_raw"].apply(_parse_periodo)
    data["año"]       = data["fecha_raw"].apply(lambda x: pd.to_datetime(x).year if pd.notna(x) else None)
    data["trimestre"] = data["fecha_raw"].apply(_get_q)
    data = data.dropna(subset=["periodo"])
    data = data[(data["periodo"] >= PERIODO_INICIO) & (data["periodo"] <= PERIODO_FIN)].copy()
    all_map = {**amap, **pmap}
    cols = ["periodo", "año", "trimestre"]
    for k, v in all_map.items():
        kn = " ".join(k.split())
        if kn in data.columns and v not in cols:
            data[v] = pd.to_numeric(data[kn], errors="coerce")
            cols.append(v)
    return data[cols].reset_index(drop=True)
 
def build_panel(filepath, label="balances"):
    """Pasos 1 y 2: construye panel de balances u operaciones."""
    print(f"\n--- {label.upper()} ---")
    es_op = "operac" in label.lower()
    dfs = []
    for sheet, (code, name) in SECTOR_MAP.items():
        if sheet == "Hogares":
            pmap = OP_PASIVOS_HOG if es_op else PASIVOS_MAP_HOGARES
        elif sheet == "Empresas_No_Financieras":
            pmap = OP_PASIVOS_ENF if es_op else PASIVOS_MAP_ENF
        else:
            pmap = OP_PASIVOS_MAP if es_op else PASIVOS_MAP
        amap = OP_ACTIVOS_MAP if es_op else ACTIVOS_MAP
        df = _read_sector_sheet(filepath, sheet, amap, pmap)
        df["sector_codigo"] = code
        df["sector_nombre"] = name
        dfs.append(df)
        print(f"  ✓ {sheet:30s} {len(df)} trimestres")
    panel = pd.concat(dfs, ignore_index=True)
    panel["_ord"] = panel["sector_codigo"].map({s: i for i, s in enumerate(SECTOR_ORDER)})
    panel = panel.sort_values(["periodo", "_ord"]).drop(columns=["_ord"])
 
    # Variables derivadas
    for cp, lp, tot in [
        ("act_titulos_cp",   "act_titulos_lp",   "act_titulos_total_calc"),
        ("act_prestamos_cp", "act_prestamos_lp",  "act_prestamos_total_calc"),
        ("act_depositos_vista","act_depositos_otros","act_depositos_total_calc"),
        ("pas_titulos_cp",   "pas_titulos_lp",   "pas_titulos_total_calc"),
        ("pas_prestamos_cp", "pas_prestamos_lp",  "pas_prestamos_total_calc"),
        ("pas_depositos_vista","pas_depositos_otros","pas_depositos_total_calc"),
    ]:
        if {cp, lp}.issubset(panel.columns):
            panel[tot] = panel[cp].fillna(0) + panel[lp].fillna(0)
        elif cp.replace("_cp","_total") in panel.columns and tot not in panel.columns:
            panel[tot] = panel[cp.replace("_cp","_total")].fillna(0)
 
    if {"act_titulos_total_calc","act_prestamos_total_calc"}.issubset(panel.columns):
        panel["act_credito_intermediado"] = (
            panel["act_titulos_total_calc"].fillna(0) +
            panel["act_prestamos_total_calc"].fillna(0)
        )
 
    cuotas = [c for c in ["act_cuotas_fmm","act_cuotas_fnm"] if c in panel.columns]
    base   = [c for c in ["act_titulos_total_calc","act_prestamos_total_calc",
                           "act_depositos_total_calc"] if c in panel.columns]
    if base:
        panel["act_sin_acciones"] = sum(panel[c].fillna(0) for c in base + cuotas)
 
    if {"act_total","pas_total"}.issubset(panel.columns):
        panel["activo_proxy"] = panel["act_total"].fillna(panel["pas_total"])
 
    print(f"  → {panel.shape[0]:,} obs × {panel.shape[1]} vars | {panel['periodo'].min()}→{panel['periodo'].max()}")
    return panel
 
# ============================================================
# FUNCIONES — PASO 3
# ============================================================
 
def build_matriz_panel():
    """Paso 3: procesa la Matriz de Exposiciones who-to-whom."""
    print("\n--- MATRIZ WHO-TO-WHOM ---")
    df = pd.read_excel(FILE_MATRIZ, sheet_name="4_Bdatos", header=0)
    df = df.rename(columns={
        "AÑO":"año","TRIM":"trimestre","PERIODO":"periodo_decimal",
        "CUENTA":"tipo_cuenta","C_SECTOR_ACTIVO":"sector_activo_codigo",
        "N_SECTOR_ACTIVO":"sector_activo_nombre","INSTRUMENTO":"instrumento",
        "C_SECTOR_PASIVO":"sector_pasivo_codigo","N_SECTOR_PASIVO":"sector_pasivo_nombre",
        "DATO":"valor",
    })
    df["periodo"]     = df["año"].astype(str) + "T" + df["trimestre"].astype(str)
    df["tipo_cuenta"] = df["tipo_cuenta"].str.strip()
 
    # Nombres canónicos de instrumentos (fuente única de verdad para todo el pipeline)
    instr_map = {
        "Cuotas MM":    "Cuotas FMM",
        "Cuotas NM":    "Cuotas FNM",
        "Depósitos":    "Depositos",
        "Dinero legal": "Dinero legal",
        "Préstamos":    "Prestamos",
        "Títulos":      "Titulos deuda",
    }
    df["instrumento"] = df["instrumento"].map(instr_map).fillna(df["instrumento"])
 
    codigo_map = {
        "S11":"S11","S121":"S121","S122":"S122","S123":"S123","S124":"S124",
        "S125+S126":"S125_S126","S128":"S128","S129":"S129","S13":"S13",
        "S14+S15":"S14_S15","S2":"S2",
    }
    df["sector_activo_codigo"] = df["sector_activo_codigo"].map(codigo_map).fillna(df["sector_activo_codigo"])
    df["sector_pasivo_codigo"] = df["sector_pasivo_codigo"].map(codigo_map).fillna(df["sector_pasivo_codigo"])
    df["valor"] = pd.to_numeric(df["valor"], errors="coerce")
    df = df[(df["periodo"] >= PERIODO_INICIO) & (df["periodo"] <= PERIODO_FIN)].copy()
 
    cols = ["periodo","año","trimestre","tipo_cuenta","sector_activo_codigo",
            "sector_activo_nombre","sector_pasivo_codigo","sector_pasivo_nombre",
            "instrumento","valor"]
    df_saldos = df[df["tipo_cuenta"] == "Bce Final"].copy()
    df_op     = df[df["tipo_cuenta"] == "Financiera"].copy()
    for d in [df, df_saldos, df_op]:
        d.sort_values(["periodo","instrumento","sector_activo_codigo"], inplace=True)
 
    print(f"  ✓ Total: {len(df):,} | Saldos: {len(df_saldos):,} | Operaciones: {len(df_op):,}")
    print(f"  ✓ Instrumentos: {sorted(df['instrumento'].unique())}")
    return df[cols], df_saldos[cols], df_op[cols]
 
# ============================================================
# FUNCIONES — PASOS 4–9 (todos desde panel_matriz_saldos / panel_matriz_operaciones)
# ============================================================
 
def build_red(df_saldos):
    """Paso 4: red financiera sectorial sin Dinero legal."""
    df = df_saldos[df_saldos["instrumento"].isin(INSTR_RED)].copy()
    # Clip negativos: errores de redondeo del proceso de conciliación BCCh
    # (máx absoluto: -0.000285 MMMCL — ver diagnóstico en notas del proyecto)
    df["valor"] = df["valor"].clip(lower=0)
    df = df.sort_values(["periodo","instrumento","sector_activo_codigo","sector_pasivo_codigo"]).reset_index(drop=True)
    print(f"  ✓ Red: {len(df):,} obs | negativos post-clip: {(df['valor']<0).sum()}")
    return df
 
def build_hhi(df_saldos):
    """Pasos 5–6: HHI de fuentes de financiamiento del sector real."""
    df_base = df_saldos[
        (df_saldos["instrumento"].isin(INSTR_HHI)) &
        (df_saldos["sector_pasivo_codigo"].isin(SECTOR_REAL))
    ].copy()
    df_base["valor"] = df_base["valor"].clip(lower=0)
    df_base["clasificacion_acreedor"] = df_base["sector_activo_codigo"].map(CLASIF_MAP)
    df_base["flag_sf_acreedor"] = df_base["sector_activo_codigo"].isin(SF)
 
    # HHI serie
    df_agg = df_base.groupby(
        ["periodo","año","trimestre","sector_pasivo_codigo","sector_activo_codigo","clasificacion_acreedor"]
    )["valor"].sum().reset_index()
    tot = df_agg.groupby(["periodo","sector_pasivo_codigo"])["valor"].sum().reset_index().rename(columns={"valor":"total_credito"})
    df_agg = df_agg.merge(tot, on=["periodo","sector_pasivo_codigo"])
    df_agg["part"]    = df_agg["valor"] / df_agg["total_credito"]
    df_agg["part_sq"] = df_agg["part"] ** 2
 
    df_serie = df_agg.groupby(
        ["periodo","año","trimestre","sector_pasivo_codigo","total_credito"]
    ).agg(hhi=("part_sq", lambda x: x.sum()*10000), n_acreedores=("sector_activo_codigo","count")).reset_index()
 
    for clase, col in [("Bancario","part_bancario"),("NBFI","part_nbfi")]:
        tmp = df_agg[df_agg["clasificacion_acreedor"]==clase].groupby(
            ["periodo","sector_pasivo_codigo"]
        )["part"].sum().reset_index().rename(columns={"part":col})
        df_serie = df_serie.merge(tmp, on=["periodo","sector_pasivo_codigo"], how="left")
 
    df_serie["part_bancario"] = df_serie["part_bancario"].fillna(0)
    df_serie["part_nbfi"]     = df_serie["part_nbfi"].fillna(0)
    df_serie["sector_nombre"] = df_serie["sector_pasivo_codigo"].map(NOMBRE_MAP)
    df_serie = df_serie.sort_values(["sector_pasivo_codigo","periodo"]).reset_index(drop=True)
 
    print(f"  ✓ HHI base: {len(df_base):,} obs | HHI serie: {len(df_serie):,} obs ({df_serie['periodo'].nunique()} trimestres)")
    return df_base, df_serie
 
def build_is(df_red, panel_bal):
    """
    Paso 7: IS Upper-Worms adaptado + exposición bilateral NBFI→Bancos.
 
    ISPasivo_i,t = sum_{j∈SF}(e_{ji,t}) / ASF_t
        → fracción del activo SF expuesta ante default de i como deudor
 
    ISActivo_i,t = sum_{j∈SF}(e_{ij,t}) / ASF_t
        → fracción del activo SF que depende de i como fuente de financiamiento
 
    Denominador: ASF_t = sum_{i∈SF}(activo_proxy_i,t)
    BCCh (S121) incluido en ASF_t pero excluido de NBFI y del VAR
    Diebold-Yılmaz (comportamiento exógeno a la dinámica financiera privada)
 
    exp_nbfi_en_bancos_i,t   = activos del sector NBFI i en instrumentos bancarios
    ratio_exp_nbfi_bancos_i,t = exp_nbfi_en_bancos_i,t / activo_proxy_i,t
        → concentración de portafolio del NBFI respecto al banco
        → complementa IS: no solo los NBFI financian a bancos (IS_activo),
          sino que concentran su propio portafolio en ellos
        → NaN para sectores no-NBFI (no aplica conceptualmente)
 
    Nota FP (S129): IS_pasivo = 0 por construcción correcta.
    Los FP no emiten pasivos en los 5 instrumentos de la red.
    Sus pasivos son cuotas de fondos de pensiones registradas como
    activos de hogares en cuenta separada, fuera de la red.
    Coherente con la singularidad NBFI→Bancos: FP financian bancos
    (IS_activo > 0) pero no se financian del sistema (IS_pasivo = 0).
    """
    # ── ASF_t ─────────────────────────────────────────────────
    asf = panel_bal[panel_bal["sector_codigo"].isin(SF)].groupby(
        "periodo")["activo_proxy"].sum().reset_index().rename(
        columns={"activo_proxy":"ASF_t"})
 
    # ── IS pasivo ──────────────────────────────────────────────
    is_p = df_red[df_red["sector_activo_codigo"].isin(SF)].groupby(
        ["periodo","sector_pasivo_codigo"]
    )["valor"].sum().reset_index().rename(
        columns={"sector_pasivo_codigo":"sector_codigo","valor":"exposicion_pasivo"})
    is_p = is_p.merge(asf, on="periodo")
    is_p["IS_pasivo"] = is_p["exposicion_pasivo"] / is_p["ASF_t"]
 
    # ── IS activo ──────────────────────────────────────────────
    is_a = df_red[df_red["sector_pasivo_codigo"].isin(SF)].groupby(
        ["periodo","sector_activo_codigo"]
    )["valor"].sum().reset_index().rename(
        columns={"sector_activo_codigo":"sector_codigo","valor":"exposicion_activo"})
    is_a = is_a.merge(asf, on="periodo")
    is_a["IS_activo"] = is_a["exposicion_activo"] / is_a["ASF_t"]
 
    # ── Exposición bilateral NBFI → Bancos (denominador: red) ───
    # Numerador: activos del sector NBFI i en instrumentos bancarios
    # Denominador: activo total del sector i desde la red (suma de
    #   todas sus exposiciones activas en todos los instrumentos)
    # Consistencia interna: numerador y denominador misma fuente
    # NaN para sectores no-NBFI (no aplica conceptualmente)
    act_nbfi_red = df_red[
        df_red["sector_activo_codigo"].isin(NBFI)
    ].groupby(["periodo","sector_activo_codigo"])["valor"].sum().reset_index().rename(
        columns={"sector_activo_codigo":"sector_codigo",
                 "valor":"activo_red_nbfi_sector"})
    exp_nb = df_red[
        (df_red["sector_activo_codigo"].isin(NBFI)) &
        (df_red["sector_pasivo_codigo"].isin(BANCARIO))
    ].groupby(["periodo","sector_activo_codigo"])["valor"].sum().reset_index().rename(
        columns={"sector_activo_codigo":"sector_codigo",
                 "valor":"exp_nbfi_en_bancos"})
    exp_nb = act_nbfi_red.merge(exp_nb, on=["periodo","sector_codigo"], how="left")
    exp_nb["exp_nbfi_en_bancos"]    = exp_nb["exp_nbfi_en_bancos"].fillna(0)
    exp_nb["ratio_exp_nbfi_bancos"] =         exp_nb["exp_nbfi_en_bancos"] / exp_nb["activo_red_nbfi_sector"]
 
    # ── Exposición NBFI agregado → Bancos ────────────────────────
    # Variable de sistema: un valor por período
    agg_exp = exp_nb.groupby("periodo")["exp_nbfi_en_bancos"].sum().reset_index()
    agg_act = exp_nb.groupby("periodo")["activo_red_nbfi_sector"].sum().reset_index().rename(
        columns={"activo_red_nbfi_sector":"activo_nbfi_red_total"})
    nbfi_agg = agg_exp.merge(agg_act, on="periodo")
    nbfi_agg["ratio_exp_nbfi_bancos_agg"] =         nbfi_agg["exp_nbfi_en_bancos"] / nbfi_agg["activo_nbfi_red_total"]
 
    # ── Exposición bilateral Bancos → NBFI (denominador: red) ────
    # En economías avanzadas este ratio es alto (bancos financian NBFI)
    # En Chile se espera bajo → confirma singularidad NBFI→Bancos
    # NaN para sectores no-bancarios (no aplica conceptualmente)
    act_bancos_red = df_red[
        df_red["sector_activo_codigo"].isin(BANCARIO)
    ].groupby(["periodo","sector_activo_codigo"])["valor"].sum().reset_index().rename(
        columns={"sector_activo_codigo":"sector_codigo",
                 "valor":"activo_bancos_red"})
    exp_bn = df_red[
        (df_red["sector_activo_codigo"].isin(BANCARIO)) &
        (df_red["sector_pasivo_codigo"].isin(NBFI))
    ].groupby(["periodo","sector_activo_codigo"])["valor"].sum().reset_index().rename(
        columns={"sector_activo_codigo":"sector_codigo",
                 "valor":"exp_bancos_en_nbfi"})
    exp_bn = act_bancos_red.merge(exp_bn, on=["periodo","sector_codigo"], how="left")
    exp_bn["exp_bancos_en_nbfi"]    = exp_bn["exp_bancos_en_nbfi"].fillna(0)
    exp_bn["ratio_exp_bancos_nbfi"] =         exp_bn["exp_bancos_en_nbfi"] / exp_bn["activo_bancos_red"]
 
    # ── Universo completo sectores × períodos ──────────────────
    periodos = sorted(df_red["periodo"].unique())
    idx = pd.MultiIndex.from_product(
        [periodos, list(NOMBRE_MAP.keys())], names=["periodo","sector_codigo"])
    df_is = pd.DataFrame(index=idx).reset_index()
    df_is = df_is.merge(
        is_p[["periodo","sector_codigo","exposicion_pasivo","ASF_t","IS_pasivo"]],
        on=["periodo","sector_codigo"], how="left")
    df_is = df_is.merge(
        is_a[["periodo","sector_codigo","exposicion_activo","IS_activo"]],
        on=["periodo","sector_codigo"], how="left")
    # Merge exposición NBFI por sector
    df_is = df_is.merge(
        exp_nb[["periodo","sector_codigo","activo_red_nbfi_sector",
                "exp_nbfi_en_bancos","ratio_exp_nbfi_bancos"]],
        on=["periodo","sector_codigo"], how="left")
    # Merge exposición Bancos→NBFI
    df_is = df_is.merge(
        exp_bn[["periodo","sector_codigo","activo_bancos_red",
                "exp_bancos_en_nbfi","ratio_exp_bancos_nbfi"]],
        on=["periodo","sector_codigo"], how="left")
    # Merge NBFI agregado (variable de sistema)
    df_is = df_is.merge(
        nbfi_agg[["periodo","activo_nbfi_red_total","ratio_exp_nbfi_bancos_agg"]],
        on="periodo", how="left")
 
    # IS = 0 para sectores sin exposición en la red
    for col in ["exposicion_pasivo","IS_pasivo","exposicion_activo","IS_activo"]:
        df_is[col] = df_is[col].fillna(0)
 
    # Exposición NBFI: 0 para NBFI sin exposición bancaria, NaN para no-NBFI
    for col in ["activo_red_nbfi_sector","exp_nbfi_en_bancos","ratio_exp_nbfi_bancos"]:
        df_is.loc[df_is["sector_codigo"].isin(NBFI), col] = \
            df_is.loc[df_is["sector_codigo"].isin(NBFI), col].fillna(0)
    # Exposición Bancos: 0 para bancario sin exposición NBFI, NaN para no-bancario
    for col in ["activo_bancos_red","exp_bancos_en_nbfi","ratio_exp_bancos_nbfi"]:
        df_is.loc[df_is["sector_codigo"].isin(BANCARIO), col] = \
            df_is.loc[df_is["sector_codigo"].isin(BANCARIO), col].fillna(0)
 
    asf_dict = asf.set_index("periodo")["ASF_t"].to_dict()
    df_is["ASF_t"]         = df_is["periodo"].map(asf_dict)
    df_is["sector_nombre"] = df_is["sector_codigo"].map(NOMBRE_MAP)
    df_is["año"]           = df_is["periodo"].str[:4].astype(int)
    df_is["trimestre"]     = df_is["periodo"].str[-1].astype(int)
    df_is = df_is.sort_values(["sector_codigo","periodo"]).reset_index(drop=True)
 
    print(f"  ✓ IS: {len(df_is):,} obs × {df_is.shape[1]} vars")
    print(f"  ✓ NaN IS_pasivo: {df_is['IS_pasivo'].isna().sum()} | NaN IS_activo: {df_is['IS_activo'].isna().sum()}")
    print(f"  ✓ IS_pasivo max: {df_is['IS_pasivo'].max():.4f} | IS_activo max: {df_is['IS_activo'].max():.4f}")
    print(f"  ✓ ratio_exp_nbfi_bancos — NBFI NaN: {df_is[df_is['sector_codigo'].isin(NBFI)]['ratio_exp_nbfi_bancos'].isna().sum()}")
    print(f"  ✓ ratio_exp_bancos_nbfi — Bancos NaN: {df_is[df_is['sector_codigo'].isin(BANCARIO)]['ratio_exp_bancos_nbfi'].isna().sum()}")
    return df_is

def build_dependencia_fondeo(df_red):
    """
    Paso 8: Índice de dependencia de fondeo bilateral.

    D^{fondeo}_{j<-i,t} = e_{ij,t} / P^{j}_{red,t}

    donde:
        e_{ij,t}        = exposición del sector i al sector j en t
                          (activo de i, pasivo de j)
        P^{j}_{red,t}   = sum_k e_{kj,t}
                          pasivo total de j en la red en t (suma sobre todos los
                          acreedores k de j en los 5 instrumentos de la red)

    Pregunta que responde:
        ¿qué fracción del fondeo total del sector j proviene del sector i
        como financiador?

    Diferencia con IS Upper-Worms (Paso 7):
        - IS^{deudor}_i e IS^{acreedor}_i son métricas SECTORIALES AGREGADAS
          (un valor por sector y período), normalizadas por el activo total
          del sistema financiero A^{SF}_t.
        - D^{fondeo}_{j<-i} es BILATERAL (un valor por par (i,j) y período),
          normalizado por el pasivo total del sector deudor j en la red.

    Conexión con HHI (Paso 5–6):
        HHI_{j,t} = sum_i (D^{fondeo}_{j<-i,t})^2 * 10000
        El HHI es el momento de segundo orden de D^{fondeo} sobre el conjunto
        de acreedores de j. La construcción del HHI en build_hhi y la de
        D^{fondeo} acá son consistentes por diseño.

    Estructura del output (panel_dependencia_fondeo):
        - Granularidad: par (i,j) × período (sumado sobre los 5 instrumentos
          de la red).
        - Solo registra pares con vínculo activo: e_{ij,t} > 0
        - Si quisieras incluir todos los pares (incluyendo D=0), cambiar el
          filtro `df_pos` por el merge con el universo completo.

    Convención de notación:
        - "deudor" = j = sector que se financia
        - "acreedor" = i = sector que financia
        - Lectura natural: "el deudor j recibe fondeo desde el acreedor i"
    """
    # ── Numerador: exposición bilateral e_{ij,t} ──────────────
    # En df_red, sector_activo es el acreedor (i) y sector_pasivo es el deudor (j).
    # Sumamos sobre instrumentos para tener exposición total bilateral en la red.
    df_bil = df_red.groupby(
        ["periodo","año","trimestre","sector_activo_codigo","sector_pasivo_codigo"]
    )["valor"].sum().reset_index().rename(columns={"valor":"exp_ij"})

    # Renombrar para claridad semántica
    df_bil = df_bil.rename(columns={
        "sector_activo_codigo": "acreedor_codigo",   # i: financiador
        "sector_pasivo_codigo": "deudor_codigo",     # j: deudor
    })

    # ── Denominador: pasivo total del deudor j en la red ──────
    # P^{j}_{red,t} = sum_k e_{kj,t} = suma de todas las exposiciones que tienen
    # como sector pasivo a j, sobre todos los acreedores k y todos los instrumentos.
    pas_red_j = df_red.groupby(
        ["periodo","sector_pasivo_codigo"]
    )["valor"].sum().reset_index().rename(columns={
        "sector_pasivo_codigo": "deudor_codigo",
        "valor": "pasivo_red_j",
    })

    # ── Construcción del índice ───────────────────────────────
    df_dep = df_bil.merge(pas_red_j, on=["periodo","deudor_codigo"], how="left")

    # Filtro de vínculos activos: solo pares (i,j) con exposición positiva.
    # Esto evita ratios indefinidos y permite contar n_acreedores por j.
    df_pos = df_dep[df_dep["exp_ij"] > 0].copy()

    # D^{fondeo}_{j<-i,t}
    # Si pasivo_red_j == 0 el sector j no tiene fondeo en la red en t y el ratio
    # es indefinido. En ese caso df_pos ya está vacío para ese j (no hay e_{ij}>0
    # con denominador 0 a menos que haya inconsistencia en los datos).
    df_pos["D_fondeo"] = df_pos["exp_ij"] / df_pos["pasivo_red_j"]

    # ── Agregar nombres y clasificación para análisis posterior ─
    df_pos["acreedor_nombre"] = df_pos["acreedor_codigo"].map(NOMBRE_MAP)
    df_pos["deudor_nombre"]   = df_pos["deudor_codigo"].map(NOMBRE_MAP)
    df_pos["acreedor_clasif"] = df_pos["acreedor_codigo"].map(CLASIF_MAP)
    df_pos["deudor_clasif"]   = df_pos["deudor_codigo"].map(CLASIF_MAP)

    # ── Reordenar columnas y ordenar filas ────────────────────
    cols = ["periodo","año","trimestre",
            "acreedor_codigo","acreedor_nombre","acreedor_clasif",
            "deudor_codigo","deudor_nombre","deudor_clasif",
            "exp_ij","pasivo_red_j","D_fondeo"]
    df_pos = df_pos[cols].sort_values(
        ["periodo","deudor_codigo","D_fondeo"],
        ascending=[True, True, False]
    ).reset_index(drop=True)

    # ── Validaciones ──────────────────────────────────────────
    # Suma de D_fondeo sobre acreedores de un mismo j en t debería ser ≈ 1
    # (es la cuota total de financiamiento del deudor j en la red)
    check    = df_pos.groupby(["periodo","deudor_codigo"])["D_fondeo"].sum()
    desv_max = (check - 1).abs().max()

    print(f"  ✓ Dependencia de fondeo: {len(df_pos):,} obs (pares activos)")
    print(f"  ✓ Pares únicos: {df_pos.groupby(['acreedor_codigo','deudor_codigo']).ngroups}")
    print(f"  ✓ Desviación máx |sum_i D_fondeo - 1| = {desv_max:.2e} (debería ≈ 0)")
    print(f"  ✓ D_fondeo: min={df_pos['D_fondeo'].min():.4f} | "
          f"mediana={df_pos['D_fondeo'].median():.4f} | "
          f"max={df_pos['D_fondeo'].max():.4f}")

    return df_pos


def _load_uf_trimestral():
    """
    Auxiliar: carga la serie de UF al cierre de cada trimestre desde el archivo
    xlsx oficial del BCCh.

    Formato de entrada (FILE_UF):
        Archivo Excel "UF_IVP_UTM_F.xlsx" del Banco Central de Chile
        (https://si3.bcentral.cl, sección Indicadores de reajustabilidad).

        Hoja "Cuadro":
          Fila 0: Título "Indicadores de reajustabilidad, fin de mes"
          Fila 2: Headers ("Periodo" | "1. Unidad de fomento (UF)" | ...)
          Fila 3+: Datos. Columna A = fecha (YYYY-MM-01 datetime),
                          Columna B = UF en pesos.

        El BCCh registra el dato mensual al último día del mes. Cuando el
        archivo se descarga filtrado trimestralmente, los meses presentes son
        marzo (cierre T1), junio (cierre T2), septiembre (cierre T3),
        diciembre (cierre T4). Este loader hace ese mapeo automáticamente.

    Mapeo mes → trimestre:
        mes  3 (marzo)      → T1
        mes  6 (junio)      → T2
        mes  9 (septiembre) → T3
        mes 12 (diciembre)  → T4
        Otros meses se descartan con warning (no debería pasar si el archivo
        ya viene filtrado trimestralmente).

    Comportamiento ante archivo ausente o malformado:
        Retorna None y emite warning. build_notional_stocks detecta este caso
        y construye solo la versión nominal del notional.

    Validaciones aplicadas:
        - Hoja "Cuadro" presente
        - Columnas Periodo y UF identificables
        - UF estrictamente positiva
        - Sin valores nulos en el período del estudio
        - Cobertura ≥ 91 trimestres (período del estudio: 2003T1-2025T3)
    """
    if not FILE_UF.exists():
        print(f"  ⚠ Archivo de UF no encontrado: {FILE_UF.name}")
        print(f"     → Se construirá solo la versión nominal del notional.")
        print(f"     → Para incluir deflactor, descargar UF_IVP_UTM_F.xlsx")
        print(f"       desde BCCh y guardarlo en {INPUT_DIR}")
        return None

    try:
        # Leer hoja Cuadro con headers en fila 2 (índice 2) → datos desde fila 3
        df = pd.read_excel(FILE_UF, sheet_name="Cuadro", header=2)
    except Exception as e:
        print(f"  ⚠ Error al leer {FILE_UF.name}: {e}")
        print(f"     → Se construirá solo la versión nominal del notional.")
        return None

    # La columna A se llama "Periodo" y la columna B contiene la UF.
    # El header exacto de la columna UF puede variar ("1. Unidad de fomento (UF)",
    # "Unidad de fomento (UF)", etc.), así que la identificamos por posición.
    if df.shape[1] < 2:
        print(f"  ⚠ {FILE_UF.name} no tiene la estructura esperada (necesita 2+ columnas).")
        print(f"     → Se construirá solo la versión nominal del notional.")
        return None

    df = df.rename(columns={df.columns[0]: "fecha_raw", df.columns[1]: "UF"})
    df = df[["fecha_raw","UF"]].copy()

    # Parseo de fechas: el BCCh las provee como datetime YYYY-MM-01
    df["fecha"] = pd.to_datetime(df["fecha_raw"], errors="coerce")
    df["UF"]    = pd.to_numeric(df["UF"], errors="coerce")
    df = df.dropna(subset=["fecha","UF"])

    # Mapeo mes → trimestre (solo meses de cierre de trimestre)
    mes_to_q = {3: 1, 6: 2, 9: 3, 12: 4}
    df["mes"] = df["fecha"].dt.month
    df_q = df[df["mes"].isin(mes_to_q.keys())].copy()
    n_descartados = len(df) - len(df_q)
    if n_descartados > 0:
        print(f"  ⚠ Descartadas {n_descartados} obs en meses no-trimestrales "
              f"(esperado 0 si el archivo viene filtrado trimestralmente).")

    df_q["trimestre"] = df_q["mes"].map(mes_to_q)
    df_q["año"]       = df_q["fecha"].dt.year
    df_q["periodo"]   = df_q["año"].astype(str) + "T" + df_q["trimestre"].astype(str)

    # Filtros finales: período del estudio y UF positiva
    df_q = df_q[(df_q["periodo"] >= PERIODO_INICIO) & (df_q["periodo"] <= PERIODO_FIN)]
    df_q = df_q[df_q["UF"] > 0].copy()

    n_obs = len(df_q)
    n_esperado = 91   # 2003T1 a 2025T3
    if n_obs < n_esperado:
        print(f"  ⚠ Serie de UF tiene {n_obs} obs (esperadas {n_esperado}).")
        print(f"     Se procederá con los trimestres disponibles; los demás")
        print(f"     quedarán sin valor real.")

    out = df_q[["periodo","UF"]].sort_values("periodo").reset_index(drop=True)
    print(f"  ✓ Serie UF cargada: {n_obs} trimestres | "
          f"UF[{out['periodo'].iloc[0]}]={out['UF'].iloc[0]:.2f} → "
          f"UF[{out['periodo'].iloc[-1]}]={out['UF'].iloc[-1]:.2f}")
    return out


def build_notional_stocks(df_op_m, df_red, uf=None):
    """
    Paso 9: Notional stocks por sector e instrumento, en versión nominal y real.

    DOS ARREGLOS COMPLEMENTARIOS, INDEPENDIENTES:

    (a) Notional à la Holm-Hadulla et al. (2023):
        Resuelve el efecto precio (revalorizaciones de mercado).
        El stock observado en saldos = cantidad × precio en t. Sus variaciones
        mezclan decisiones activas de portafolio con revalorizaciones. Las
        operaciones financieras (df_op_m), en cambio, registran solo
        transacciones a precio del momento, sin revalorizar.

            A_{i,k,t}^{nominal} = A_{i,k,t0}^{observado}
                                  + sum_{tau=t0+1}^{t} F_{i,k,tau}^{nominal}

    (b) Deflactado por UF similar a working paper de Jara y deflactación de Antonakakis et al. (2015):
        Resuelve el efecto inflación. Sin deflactar, el VAR D-Y detecta
        spillovers espurios por inflación común que afecta a todos los
        sectores nominales simultáneamente. La UF se ajusta diariamente por
        IPC, así que dividir por UF es equivalente a deflactar por IPC.

            A_{i,k,t}^{real} = A_{i,k,t0}^{nominal}/UF_{t0}
                               + sum_{tau=t0+1}^{t} F_{i,k,tau}^{nominal}/UF_tau

        Cada flujo se deflacta por la UF de su propio período → respeta el
        principio de que cada operación se valora a precios de su momento.
        El saldo base se deflacta por UF inicial → ancla la serie en nivel
        real correcto.

    PROPIEDADES DE LA CONSTRUCCIÓN:
        Ambos arreglos son necesarios y resuelven problemas distintos:
        (a) sin (b) → variable nominal contaminada por inflación común.
        (b) sin (a) → variable contaminada por efecto precio.
        (a) + (b)   → variable que captura solo decisiones activas de
                      portafolio, expresadas en poder adquisitivo constante.

        La primera diferencia del notional real es el flujo real del trimestre:
            A_{i,k,t}^{real} - A_{i,k,t-1}^{real} = F_{i,k,t}^{nominal}/UF_t

    REFERENCIAS:
        Holm-Hadulla, Mazelis & Rast (2023), "Bank and non-bank balance sheet
        responses to monetary policy shocks", Economics Letters 222.

        Antonakakis, Breitenlechner & Scharler (2015), "Business cycle and
        financial cycle spillovers in the G7 countries", Quarterly Review of
        Economics and Finance 58, 154-162. Aplican Diebold-Yilmaz a credit
        growth y GDP growth trimestrales en términos reales.

    USO POSTERIOR:
        Las series y_{i,t} = Δ log A_{i,t}^{real, agregado} (donde agregado =
        suma sobre los 5 instrumentos) son las variables del VAR D-Y. Excluye
        S121 (Banco Central) por exogeneidad de su comportamiento.

    GRANULARIDAD DEL OUTPUT:
        df_notional       → sector × instrumento × período (panel largo)
                            con A_t0_nominal, A_t0_real, F_t_nominal, F_t_real,
                            A_notional_nominal, A_notional_real, UF_t.
                            Si uf=None: solo columnas nominales.
        df_notional_agg   → sector × período (suma sobre instrumentos)
        df_var_dy         → sector × período (sin S121) con log y Δlog para VAR

    ARGS:
        df_op_m: panel de operaciones bilaterales por instrumento (Paso 3)
        df_red:  panel de saldos bilaterales en la red (Paso 4)
        uf:      DataFrame con columnas [periodo, UF] o None.
                 Si None: solo construye versión nominal.
    """
    # ── Saldo base nominal en t0 = 2003T1 desde df_red ────────
    # df_red tiene saldos bilaterales por instrumento. Sumamos sobre contraparte
    # para obtener el activo total del sector i en el instrumento k en t0.
    t0 = PERIODO_INICIO

    saldo_t0 = df_red[df_red["periodo"] == t0].groupby(
        ["sector_activo_codigo","instrumento"]
    )["valor"].sum().reset_index().rename(columns={
        "sector_activo_codigo":"sector_codigo",
        "valor":"A_t0_nominal",
    })

    # ── Operaciones nominales por sector e instrumento ────────
    # df_op_m está en formato bilateral por instrumento. Sumamos sobre contraparte
    # para obtener la operación total del sector i en el instrumento k en cada t.
    # Esto INCLUYE operaciones contra todos los sectores (Hogares, RoW, Gobierno,
    # etc.), no solo contra los sectores del VAR. Coherente con que la variable
    # del VAR es el comportamiento agregado de cada sector financiero.
    op_sector = df_op_m[df_op_m["instrumento"].isin(INSTR_RED)].groupby(
        ["periodo","año","trimestre","sector_activo_codigo","instrumento"]
    )["valor"].sum().reset_index().rename(columns={
        "sector_activo_codigo":"sector_codigo",
        "valor":"F_t_nominal",
    })

    # ── Universo completo: sector × instrumento × período ─────
    # Para que el cumsum esté bien definido en cada serie, garantizamos que
    # todos los pares (sector, instrumento) tengan observación en todos los
    # períodos, rellenando con F_t=0 donde no hay operación.
    periodos     = sorted(df_red["periodo"].unique())
    sectores_sf  = SECTOR_INTERES
    instrumentos = INSTR_RED

    idx = pd.MultiIndex.from_product(
        [periodos, sectores_sf, instrumentos],
        names=["periodo","sector_codigo","instrumento"]
    )
    df_notional = pd.DataFrame(index=idx).reset_index()

    # Merge con operaciones nominales (rellena F_t=0 donde no hay observación)
    df_notional = df_notional.merge(
        op_sector[["periodo","sector_codigo","instrumento","F_t_nominal"]],
        on=["periodo","sector_codigo","instrumento"],
        how="left"
    )
    df_notional["F_t_nominal"] = df_notional["F_t_nominal"].fillna(0)

    # Merge con saldo base nominal
    df_notional = df_notional.merge(
        saldo_t0, on=["sector_codigo","instrumento"], how="left"
    )
    df_notional["A_t0_nominal"] = df_notional["A_t0_nominal"].fillna(0)

    # ── Acumulación nominal (Holm-Hadulla puro) ───────────────
    # A^{nominal}_{i,k,t} = A_{t0}^{nominal} + cumsum F^{nominal} desde t0+1
    # En t = t0: A_nominal = A_t0_nominal (operación inicial no se acumula)
    # En t > t0: A_nominal = A_t0_nominal + sum F_nominal hasta t
    df_notional = df_notional.sort_values(
        ["sector_codigo","instrumento","periodo"]
    ).reset_index(drop=True)

    mask_t0 = df_notional["periodo"] == t0
    df_notional["F_t_nominal_para_cumsum"] = df_notional["F_t_nominal"].where(~mask_t0, 0)
    df_notional["F_acum_nominal"] = df_notional.groupby(
        ["sector_codigo","instrumento"]
    )["F_t_nominal_para_cumsum"].cumsum()

    df_notional["A_notional_nominal"] = (
        df_notional["A_t0_nominal"] + df_notional["F_acum_nominal"]
    )
    df_notional = df_notional.drop(columns=["F_t_nominal_para_cumsum"])

    # ── Deflactación por UF (Antonakakis) ─────────────────────
    # Si uf disponible: agregar columnas UF_t, F_t_real, A_t0_real, A_notional_real.
    # Si uf=None: queda solo la versión nominal.
    has_uf = uf is not None and len(uf) > 0

    if has_uf:
        uf_dict = uf.set_index("periodo")["UF"].to_dict()
        UF_t0   = uf_dict.get(t0, np.nan)

        if pd.isna(UF_t0):
            print(f"  ⚠ UF en t0={t0} no disponible. No se puede deflactar el "
                  f"saldo base. Se procederá solo con versión nominal.")
            has_uf = False
        else:
            # UF del período de cada flujo
            df_notional["UF_t"] = df_notional["periodo"].map(uf_dict)

            # Deflactación de saldo base: A_t0_nominal / UF_t0 (escalar)
            df_notional["A_t0_real"] = df_notional["A_t0_nominal"] / UF_t0

            # Deflactación de cada flujo por la UF de su propio período
            # F^{real}_{i,k,tau} = F^{nominal}_{i,k,tau} / UF_tau
            # NaN-safe: si UF_t es NaN para algún período, F_real queda NaN y
            # se propaga al cumsum de ese sector-instrumento desde ese punto.
            df_notional["F_t_real"] = df_notional["F_t_nominal"] / df_notional["UF_t"]

            # Acumulación real desde t0+1
            df_notional["F_t_real_para_cumsum"] = df_notional["F_t_real"].where(~mask_t0, 0)
            df_notional["F_acum_real"] = df_notional.groupby(
                ["sector_codigo","instrumento"]
            )["F_t_real_para_cumsum"].cumsum()

            df_notional["A_notional_real"] = (
                df_notional["A_t0_real"] + df_notional["F_acum_real"]
            )
            df_notional = df_notional.drop(columns=["F_t_real_para_cumsum"])

    # ── Año, trimestre, nombre de sector ──────────────────────
    df_notional["año"]           = df_notional["periodo"].str[:4].astype(int)
    df_notional["trimestre"]     = df_notional["periodo"].str[-1].astype(int)
    df_notional["sector_nombre"] = df_notional["sector_codigo"].map(NOMBRE_MAP)

    # Orden de columnas para el output (más interpretable)
    cols_base = ["periodo","año","trimestre","sector_codigo","sector_nombre",
                 "instrumento","A_t0_nominal","F_t_nominal","F_acum_nominal",
                 "A_notional_nominal"]
    cols_real = ["UF_t","A_t0_real","F_t_real","F_acum_real","A_notional_real"]
    cols      = cols_base + (cols_real if has_uf else [])

    df_notional = df_notional[cols].sort_values(
        ["sector_codigo","instrumento","periodo"]
    ).reset_index(drop=True)

    # ── Notional agregado por sector (suma sobre instrumentos) ─
    # Sumas separadas para nominal y real (si disponible).
    agg_cols = ["A_notional_nominal"] + (["A_notional_real"] if has_uf else [])
    df_notional_agg = df_notional.groupby(
        ["periodo","año","trimestre","sector_codigo","sector_nombre"]
    )[agg_cols].sum().reset_index()

    # Renombrar para claridad: total agregado sobre instrumentos
    df_notional_agg = df_notional_agg.rename(columns={
        "A_notional_nominal": "A_notional_nominal_total",
        **({"A_notional_real": "A_notional_real_total"} if has_uf else {}),
    })
    df_notional_agg = df_notional_agg.sort_values(
        ["sector_codigo","periodo"]
    ).reset_index(drop=True)

    # ── Serie para VAR Diebold-Yilmaz ─────────────────────────
    # y_{i,t} = Δ log A_{i,t}^{real, agregado}
    # Si no hay UF: usa la versión nominal y advierte explícitamente.
    df_var_dy = df_notional_agg.copy() 
    #[ df_notional_agg["sector_codigo"] != "S121"].copy() #!= "S121" AJUSTE PARA TENER IGUAL BCCh

    if has_uf:
        var_col = "A_notional_real_total"
        log_col = "log_A_real"
        d_col   = "d_log_A_real"
    else:
        var_col = "A_notional_nominal_total"
        log_col = "log_A_nominal"
        d_col   = "d_log_A_nominal"

    # Log seguro
    df_var_dy[log_col] = np.where(
        df_var_dy[var_col] > 0,
        np.log(df_var_dy[var_col]),
        np.nan
    )
    df_var_dy = df_var_dy.sort_values(["sector_codigo","periodo"]).reset_index(drop=True)
    df_var_dy[d_col] = df_var_dy.groupby("sector_codigo")[log_col].diff()

    # ── Validaciones ──────────────────────────────────────────
    n_neg_nom  = (df_notional["A_notional_nominal"] < 0).sum()
    n_neg_real = (df_notional["A_notional_real"] < 0).sum() if has_uf else 0
    n_zero_log = df_var_dy[log_col].isna().sum()

    print(f"  ✓ Notional stocks: {len(df_notional):,} obs "
          f"({df_notional['sector_codigo'].nunique()} sectores × "
          f"{df_notional['instrumento'].nunique()} instr × "
          f"{df_notional['periodo'].nunique()} periodos)")
    print(f"  ✓ Versión nominal (Holm-Hadulla):    "
          f"max={df_notional['A_notional_nominal'].max():,.0f}")
    if has_uf:
        print(f"  ✓ Versión real (Holm-Hadulla + UF): "
              f"max={df_notional['A_notional_real'].max():,.0f} (en UF)")
        print(f"  ✓ Variable del VAR: d_log_A_real")
    else:
        print(f"  ⚠ Versión real NO construida (UF no disponible).")
        print(f"  ⚠ Variable del VAR: d_log_A_nominal — INTERPRETAR CON CAUTELA")
        print(f"     porque puede capturar comovimiento por inflación común.")
    print(f"  ✓ Serie VAR D-Y (con S121, luego en código DY se ve si se incluye): {len(df_var_dy):,} obs | "
          f"Δ log A no-nulos: {df_var_dy[d_col].notna().sum()}")
    print(f"  ⚠ Notional nominal < 0: {n_neg_nom} (esperado 0)")
    if has_uf:
        print(f"  ⚠ Notional real    < 0: {n_neg_real} (esperado 0)")

    if n_neg_nom > 0:
        casos = df_notional[df_notional["A_notional_nominal"] < 0][
            ["periodo","sector_codigo","instrumento",
             "A_t0_nominal","F_acum_nominal","A_notional_nominal"]
        ].head(10)
        print(f"  ⚠ Primeros casos con notional nominal negativo:\n{casos.to_string(index=False)}")

    return df_notional, df_notional_agg, df_var_dy

# ============================================================
# EJECUCIÓN PRINCIPAL
# ============================================================
 
def main():
    print("\n" + "="*60)
    print("PIPELINE CNSI — TESIS UC")
    print("="*60)
 
    for f in [FILE_BALANCES, FILE_OPERACIONES, FILE_MATRIZ]:
        if not f.exists():
            print(f"\n⚠️  ARCHIVO NO ENCONTRADO: {f}")
            return
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
 
    # Pasos 1–2: balances y operaciones
    panel_bal = build_panel(FILE_BALANCES, "balances")
    panel_op  = build_panel(FILE_OPERACIONES, "operaciones")
 
    # Paso 3: matriz who-to-whom
    print("\n--- MATRIZ WHO-TO-WHOM ---")
    df_mat, df_saldos, df_op_m = build_matriz_panel()
 
    # Pasos 4–7: todos desde df_saldos
    print("\n--- INDICADORES (fuente: panel_matriz_saldos) ---")
    df_red          = build_red(df_saldos)
    df_hhi_base, df_hhi_serie = build_hhi(df_saldos)
    df_is           = build_is(df_red, panel_bal)

    # Paso 8: Dependencia de fondeo bilateral (D^{fondeo}_{j<-i,t})
    df_dep_fondeo   = build_dependencia_fondeo(df_red)

    # Paso 9: Notional stocks à la Holm-Hadulla (2023) + deflactor UF (Antonakakis 2015)
    print("\n--- NOTIONAL STOCKS (fuente: panel_matriz_operaciones + saldo base + UF) ---")
    uf = _load_uf_trimestral()
    df_notional, df_notional_agg, df_var_dy = build_notional_stocks(df_op_m, df_red, uf=uf)
 
    # ── Guardar CSV ───────────────────────────────────────────
    panel_bal.to_csv(OUTPUT_DIR/"panel_balances.csv",             index=False, encoding="utf-8-sig")
    panel_op.to_csv(OUTPUT_DIR/"panel_operaciones.csv",           index=False, encoding="utf-8-sig")
    df_mat.to_csv(OUTPUT_DIR/"panel_matriz_completa.csv",         index=False, encoding="utf-8-sig")
    df_saldos.to_csv(OUTPUT_DIR/"panel_matriz_saldos.csv",        index=False, encoding="utf-8-sig")
    df_op_m.to_csv(OUTPUT_DIR/"panel_matriz_operaciones.csv",     index=False, encoding="utf-8-sig")
    df_red.to_csv(OUTPUT_DIR/"panel_red_saldos.csv",              index=False, encoding="utf-8-sig")
    df_hhi_base.to_csv(OUTPUT_DIR/"panel_hhi_fuentes.csv",        index=False, encoding="utf-8-sig")
    df_hhi_serie.to_csv(OUTPUT_DIR/"resumen_hhi_por_periodo.csv", index=False, encoding="utf-8-sig")
    df_is.to_csv(OUTPUT_DIR/"panel_is_upper_worms.csv",           index=False, encoding="utf-8-sig")
    df_dep_fondeo.to_csv(OUTPUT_DIR/"panel_dependencia_fondeo.csv",      index=False, encoding="utf-8-sig")
    df_notional.to_csv(OUTPUT_DIR/"panel_notional_stocks.csv",            index=False, encoding="utf-8-sig")
    df_notional_agg.to_csv(OUTPUT_DIR/"panel_notional_stocks_agregado.csv", index=False, encoding="utf-8-sig")
    df_var_dy.to_csv(OUTPUT_DIR/"panel_var_dy.csv",                       index=False, encoding="utf-8-sig")
 
    # Guardar exposiciones bilaterales como bases independientes
    # (subsets del panel_is para análisis y figuras específicas)
    exp_nb_sector = df_is[df_is["sector_codigo"].isin(NBFI)][
        ["periodo","año","trimestre","sector_codigo","sector_nombre",
         "activo_red_nbfi_sector","exp_nbfi_en_bancos","ratio_exp_nbfi_bancos"]
    ].dropna(subset=["activo_red_nbfi_sector"]).copy()
    exp_nb_sector.to_csv(OUTPUT_DIR/"panel_exp_nbfi_bancos_sector.csv",
                         index=False, encoding="utf-8-sig")
 
    exp_bn_df = df_is[df_is["sector_codigo"].isin(BANCARIO)][
        ["periodo","año","trimestre","sector_codigo","sector_nombre",
         "activo_bancos_red","exp_bancos_en_nbfi","ratio_exp_bancos_nbfi"]
    ].dropna(subset=["activo_bancos_red"]).copy()
    exp_bn_df.to_csv(OUTPUT_DIR/"panel_exp_bancos_nbfi.csv",
                     index=False, encoding="utf-8-sig")
 
    exp_agg = df_is[["periodo","año","trimestre",
                      "activo_nbfi_red_total","ratio_exp_nbfi_bancos_agg"]
    ].drop_duplicates("periodo").sort_values("periodo").copy()
    exp_agg.to_csv(OUTPUT_DIR/"panel_exp_nbfi_bancos_agregado.csv",
                   index=False, encoding="utf-8-sig")
 
    # ── Guardar Excel ─────────────────────────────────────────
    # Corrección: crear subcarpeta antes de guardar
    excel_dir = OUTPUT_DIR / "formato_excel"
    excel_dir.mkdir(parents=True, exist_ok=True)
 
    panel_bal.to_excel(excel_dir/"panel_balances.xlsx",             index=False, engine="openpyxl")
    panel_op.to_excel(excel_dir/"panel_operaciones.xlsx",           index=False, engine="openpyxl")
    df_mat.to_excel(excel_dir/"panel_matriz_completa.xlsx",         index=False, engine="openpyxl")
    df_saldos.to_excel(excel_dir/"panel_matriz_saldos.xlsx",        index=False, engine="openpyxl")
    df_op_m.to_excel(excel_dir/"panel_matriz_operaciones.xlsx",     index=False, engine="openpyxl")
    df_red.to_excel(excel_dir/"panel_red_saldos.xlsx",              index=False, engine="openpyxl")
    df_hhi_base.to_excel(excel_dir/"panel_hhi_fuentes.xlsx",        index=False, engine="openpyxl")
    df_hhi_serie.to_excel(excel_dir/"resumen_hhi_por_periodo.xlsx", index=False, engine="openpyxl")
    df_is.to_excel(excel_dir/"panel_is_upper_worms.xlsx",           index=False, engine="openpyxl")
    df_dep_fondeo.to_excel(excel_dir/"panel_dependencia_fondeo.xlsx",       index=False, engine="openpyxl")
    df_notional.to_excel(excel_dir/"panel_notional_stocks.xlsx",            index=False, engine="openpyxl")
    df_notional_agg.to_excel(excel_dir/"panel_notional_stocks_agregado.xlsx", index=False, engine="openpyxl")
    df_var_dy.to_excel(excel_dir/"panel_var_dy.xlsx",                       index=False, engine="openpyxl")
 
    print("\n" + "="*60)
    print("BASES GENERADAS")
    print("="*60)
    bases = [
        ("panel_balances.csv",                  panel_bal),
        ("panel_operaciones.csv",               panel_op),
        ("panel_matriz_completa.csv",           df_mat),
        ("panel_matriz_saldos.csv",             df_saldos),
        ("panel_matriz_operaciones.csv",        df_op_m),
        ("panel_red_saldos.csv",                df_red),
        ("panel_hhi_fuentes.csv",               df_hhi_base),
        ("resumen_hhi_por_periodo.csv",         df_hhi_serie),
        ("panel_is_upper_worms.csv",            df_is),
        ("panel_dependencia_fondeo.csv",        df_dep_fondeo),
        ("panel_notional_stocks.csv",           df_notional),
        ("panel_notional_stocks_agregado.csv",  df_notional_agg),
        ("panel_var_dy.csv",                    df_var_dy),
    ]
    for nombre, df in bases:
        print(f"  {nombre:<42s} {df.shape[0]:>7,} obs × {df.shape[1]:>2} vars")
 
    print("\n✅ Pipeline completado.")
 
 
if __name__ == "__main__":
    main()


PIPELINE CNSI — TESIS UC

--- BALANCES ---
  ✓ BC                             91 trimestres
  ✓ Bancos                         91 trimestres
  ✓ FMM                            91 trimestres
  ✓ FNM                            91 trimestres
  ✓ Otros                          91 trimestres
  ✓ Seguros                        91 trimestres
  ✓ FP                             91 trimestres
  ✓ Gobierno_General               91 trimestres
  ✓ Empresas_No_Financieras        91 trimestres
  ✓ Hogares                        91 trimestres
  ✓ Resto_Mundo                    91 trimestres
  → 1,001 obs × 36 vars | 2003T1→2025T3

--- OPERACIONES ---
  ✓ BC                             91 trimestres
  ✓ Bancos                         91 trimestres
  ✓ FMM                            91 trimestres
  ✓ FNM                            91 trimestres
  ✓ Otros                          91 trimestres
  ✓ Seguros                        91 trimestres
  ✓ FP                             91 trimestres
  ✓ Gobierno_